# Evaluate a trained checkpoint

Set CHECKPOINT to the best.pt produced by the training notebook. This evaluates the validation set and renders one test/validation prediction for a quick visual check.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TrayMachi/indonesia-license-plate-model.git"
REPO_DIR = Path("/content/indonesia-license-plate-model")
if not (REPO_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
%pip install -q -r /content/indonesia-license-plate-model/requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_YAML = REPO_DIR / "configs" / "dataset.yaml"
DRIVE_ROOT = Path("/content/drive/MyDrive/indonesia-license-plate-model")
CHECKPOINT = DRIVE_ROOT / "runs" / "plate-detector" / "weights" / "best.pt"
if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Update CHECKPOINT; file not found: {CHECKPOINT}")

In [ ]:
from ultralytics import YOLO

model = YOLO(str(CHECKPOINT))
metrics = model.val(data=str(DATASET_YAML), split="val", imgsz=640, plots=True)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_candidates = [
    p for split in ("test", "val")
    for p in (DRIVE_ROOT / "dataset_yolo" / "images" / split).rglob("*")
    if p.suffix.lower() in image_extensions
]
if not image_candidates:
    raise FileNotFoundError("No test or validation images found.")

prediction_dir = DRIVE_ROOT / "evaluation"
predictions = model.predict(
    source=str(image_candidates[0]),
    conf=0.25,
    imgsz=640,
    save=True,
    project=str(prediction_dir),
    name="predictions",
    exist_ok=True,
)
print("Rendered prediction:", Path(predictions[0].save_dir))

Use the saved plots and rendered prediction to spot missed plates, incorrect boxes, or label quality issues before exporting to Android.